# 第 05 课：异常处理与数据检查

**前置知识：** 第 01–04 课的列表、字典、循环、条件、函数、Path 文件读写和 JSON。  
**建议用时：** 2–3 小时，分为“识别异常”和“组合程序”两次学习。练习与复写可另用 1–2 小时。  
**配套脚本：** `lesson_05_errors_and_validation.py`。

上一课在正确输入上完成了读写。今天要回答：文件不存在、JSON 写错、字段缺失或编号越界时，程序应该怎样说明问题？

学完你应能：读懂异常类型；用 try/except 处理预期错误；用 raise ValueError 报告数据问题；把检查放在处理与保存之前。

本课自带完整函数与示例数据，只需 Python，暂时不使用 GPU。这里的字符词表仍是练习工具，尚未训练任何模型。


## 开始前：运行方式与文件约定

选择 Python 3 内核，从上到下按 Shift + Enter。分步示例按顺序执行；“完整程序”单元和三个参考答案各自可以独立运行。故障示例都已经捕获预期异常，因此整本讲义可以从头运行。到练习区先停下，独立完成后再读参考答案。

**运行全部单元不等于完成练习。** 练习起始单元只有合法的数据和注释，参考答案则可以直接运行。已保存的输出来自验证环境，重新运行后路径会变成你电脑上的路径。

所有本课数据放在当前工作目录的 `lesson_05_data` 下。它不一定是脚本所在目录；完整程序会打印路径。

| 路径 | 用途与行为 |
| --- | --- |
| `input_records.json` | 完整程序的输入；缺失时创建示例，已存在时保留 |
| `validated_records.json` | 所有记录检查通过后更新的生成结果 |
| `missing_for_demo.json` | 只用于演示缺失文件，不自动创建 |
| `generated_exercise_c/` | 练习 C 参考答案专用目录；其中的输入和输出是可重复生成的演示文件 |

本课处理小文件，输入、输出使用不同路径。首次创建示例需要当前目录可写。读取、JSON 解析或校验失败时，已有主结果不更新；普通 write_text 写盘若中途失败，则不能保证旧文件完整，原子保存留到后续工程课。


In [1]:
import json
from pathlib import Path

work_dir = Path.cwd() / "lesson_05_data"
work_dir.mkdir(exist_ok=True)
print("本课数据目录：", work_dir.resolve())


本课数据目录： D:\code\AI_LLM_and_Deep learning\大模型学习路线\lesson_05_data


## 1. 先判断：问题发生在哪一层？

| 问题 | 本课如何识别 | 修复方向 |
| --- | --- | --- |
| 读取的路径没有文件 | FileNotFoundError | 检查路径或补齐输入 |
| 文本不符合 JSON 格式 | json.JSONDecodeError | 检查引号、逗号、括号 |
| JSON 能解析，但数据不符合约定 | 我们主动 raise ValueError | 补齐字段，修正类型或编号 |

Python 源代码少冒号、缩进写错，与输入文件中的 JSON 格式错误是不同问题。把 try 写在语法错误的代码旁边，不能让这段 Python 自动变得合法。

遇到未处理的 traceback（错误回溯），先读最后一行的“异常类型与说明”，再向上找自己写的文件和行号。今天的示例会捕获预期异常，并打印更贴近任务的说明。


## 2. try/except：失败后转到对应处理分支

try 下面放可能失败的操作；except 后面写这次准备处理的异常类型。

**先预测：** 如果文件不存在，下面标为 A、B、C 的三个位置，哪些会执行？


In [2]:
missing_path = work_dir / "missing_for_demo.json"

print("A：准备读取")
try:
    demo_text = missing_path.read_text(encoding="utf-8")
    print("B：读取成功：", demo_text)
except FileNotFoundError:
    print("找不到文件，请检查路径：", missing_path.resolve())

print("C：这次演示结束")


A：准备读取
找不到文件，请检查路径： D:\code\AI_LLM_and_Deep learning\大模型学习路线\lesson_05_data\missing_for_demo.json
C：这次演示结束


文件不存在时，会打印 A、错误提示、C，跳过 try 中读取之后的 B。如果你自己创建了这个文件，这个例子就会走成功路径。

捕获异常只是决定接下来怎么处理；它没有创建文件，也没有替你纠正路径。

另一个常见写法是 `except ... as error:`，把异常保存为变量 error，便于读取说明。下一节使用它。


## 3. JSONDecodeError：定位格式错误

下面 Python 字符串本身合法，但它装着的 JSON 文本在最后一个字段后多了逗号。json.loads 解析时会失败。

error.lineno、error.colno 是 JSON 文本的行号与列号，不是 Notebook 单元编号；error.msg 是解析器的说明。这里的点号表示读取这个异常对象提供的信息。


In [3]:
bad_json = '{"text": "我爱", "token_ids": [0, 1],}'

try:
    parsed = json.loads(bad_json)
    print("这行不会执行，因为前一步解析失败。")
except json.JSONDecodeError as error:
    print("JSON 格式错误：第", error.lineno, "行，第", error.colno, "列")
    print("解析器说明：", error.msg)


JSON 格式错误：第 1 行，第 35 列
解析器说明： Illegal trailing comma before end of object


JSON 中的字符串和字段名使用双引号，列表与对象的末项后不能多一个逗号。Python 字典直接写在代码里时可以有末尾逗号，不能据此假设 JSON 也可以。

再看一个语法正确的 JSON。能解析，只说明得到了某个 Python 值，不说明它一定是我们要的“记录列表”。


In [4]:
valid_json = '{"text": "我爱", "token_ids": [0, 1]}'
parsed = json.loads(valid_json)

print("解析结果：", parsed)
print("它是列表吗？", isinstance(parsed, list))
print("它是字典吗？", isinstance(parsed, dict))


解析结果： {'text': '我爱', 'token_ids': [0, 1]}
它是列表吗？ False
它是字典吗？ True


## 4. isinstance：先确认数据类型

`isinstance(值, 类型)` 判断一个值是否属于指定类型，返回 True 或 False。本课使用 list、dict、str、int、bool。

| 代码 | 结果 | 含义 |
| --- | --- | --- |
| `isinstance([], list)` | True | 是列表，即使列表为空 |
| `isinstance("我爱", str)` | True | 是字符串 |
| `isinstance("6", int)` | False | 引号里的 6 是文字 |
| `isinstance(6.0, int)` | False | 6.0 是浮点数 |

一个容易漏掉的细节：Python 的布尔值属于整数的子类型，因此 True 也能通过 int 检查。本课的编号规则不接受布尔值，要先单独排除 bool；现在记住这一行为即可，不必先学类的继承。


In [6]:
print(isinstance([], list))
print(isinstance("我爱", str))
print(isinstance("6", int))
print(isinstance(6.0, int))
print("True 通过 int 检查：", isinstance(True, int))
print("True 是布尔值：", isinstance(True, bool))


True
True
False
False
True 通过 int 检查： True
True 是布尔值： True


## 5. raise：主动说明数据哪里不符合约定

读取文件时，Python 可以自动发现文件不存在。但“本项目要求最外层必须是列表”是我们自己定义的约定，需要自己检查。

`raise ValueError("说明")` 表示在这里报告一个数据不符合约定的问题。它会立即中断当前正常执行路径，交给匹配的 except 处理；如果外层没有处理，就会显示 traceback。

下面把“检查列表”写成一个小函数，观察函数里面的 raise 如何由调用处的 except 处理。


In [7]:
def require_list(value):
    if not isinstance(value, list):
        raise ValueError("最外层应该是列表，请检查是否缺少方括号。")
    return value

try:
    require_list({"text": "我爱"})
    print("只有检查通过，才会执行这一行。")
except ValueError as error:
    print("检查未通过：", error)

print("空列表也符合列表类型：", require_list([]))


检查未通过： 最外层应该是列表，请检查是否缺少方括号。
空列表也符合列表类型： []


## 6. 明确本课的数据约定

我们兼容上一课生成的记录，但只对当前要使用的两个字段作要求：

| 检查对象 | 规则 |
| --- | --- |
| 最外层 | list；允许空列表 |
| 每条记录 | dict |
| text | 必须存在，且是 str；允许空字符串 |
| token_ids | 必须存在，且是 list；允许空列表 |
| 每个编号 | int，排除 bool；不小于 0，且小于词表长度 |
| 额外字段 | 允许并保留，例如 line_number；本课不检查这些字段的内容 |

当前词表有 7 个元素，所以合法编号为 0–6。6 对应 `<unk>`，合法；-1、7、"6"、6.0、True 均不符合编号规则。Python 列表虽然支持负索引，本项目的 token 编号不使用负数。

**检查顺序由外到内：** 先确认是列表，再逐项确认是字典；先确认字段存在，再取值；先确认 token_ids 是列表，再循环；先确认元素是整数，再比较范围。

本课只检查结构和编号范围，**尚未检查 text 与 token_ids 是否对应**。例如 `{"text": "我", "token_ids": [1]}` 会通过检查，但编号实际会解码成“爱”。这个边界将在后面直接观察。


### 补充三种小写法，方便读下一段代码

- `"text" not in record`：字典里没有 text 这个键。
- `str(2)`：把数字 2 转成字符串，才能与说明文字拼接。
- `<` 表示小于；`>=` 表示大于或等于。

错误提示中的“第 2 条记录”从 1 开始计数；token_ids[1] 则使用从 0 开始的列表索引，指第 2 个编号。


In [8]:
record_number = 2
token_index = 1
location = "第 " + str(record_number) + " 条记录的 token_ids[" + str(token_index) + "]"
print(location)
print("7 是否超过合法编号范围？", 7 >= 7)
print("-1 是否小于 0？", -1 < 0)


第 2 条记录的 token_ids[1]
7 是否超过合法编号范围？ True
-1 是否小于 0？ True


## 7. 实现逐层检查

下面函数有两个参数：records 是待检查的数据；vocab_size 是当前词表的长度，由 len(id_to_token) 提供。

发现第一个问题就停止，并说明记录位置。检查全部通过时返回原 records，不修改输入，也不会把 "6" 自动转换成 6。

阅读时先看最外层检查，再看记录字段，最后看编号循环。暂时不要求一次背下整段代码。


In [ ]:
def validate_records(records, vocab_size):
    if not isinstance(records, list):
        raise ValueError("最外层必须是列表，例如 [{...}, {...}]")

    record_number = 1
    for record in records:
        location = "第 " + str(record_number) + " 条记录"

        if not isinstance(record, dict):
            raise ValueError(location + "必须是字典")
        if "text" not in record:
            raise ValueError(location + "缺少 text 字段")
        if "token_ids" not in record:
            raise ValueError(location + "缺少 token_ids 字段")
        if not isinstance(record["text"], str):
            raise ValueError(location + "的 text 必须是字符串")
        if not isinstance(record["token_ids"], list):
            raise ValueError(location + "的 token_ids 必须是列表")

        token_index = 0
        for token_id in record["token_ids"]:
            token_location = location + "的 token_ids[" + str(token_index) + "]"
            # Python 的 bool 也能通过 int 类型检查，因此先排除它。
            if isinstance(token_id, bool):
                raise ValueError(token_location + "不能是布尔值")
            if not isinstance(token_id, int):
                raise ValueError(token_location + "必须是整数")
            if token_id < 0:
                raise ValueError(token_location + "不能小于 0")
            if token_id >= vocab_size:
                raise ValueError(token_location + "必须小于词表大小 " + str(vocab_size))
            token_index = token_index + 1

        record_number = record_number + 1

    # 检查通过后返回原数据；这个函数不修改、不补齐、不纠正数据。
    return records


### 先预测，再运行

下面三份数据分别是：未知字符的合法编号、空记录列表、越界编号。哪一份会被拒绝？


In [ ]:
id_to_token = ["我", "爱", "学", "习", "模", "型", "<unk>"]

examples = [
    [{"text": "大", "token_ids": [6]}],
    [],
    [{"text": "我", "token_ids": [7]}],
]

for records in examples:
    try:
        checked = validate_records(records, len(id_to_token))
        print("通过：", checked)
    except ValueError as error:
        print("拒绝：", error)


## 8. 读取与解码继续使用简单函数

load_records 负责读取和解析，出错时交给调用它的外层处理。decode 是上一课的函数；本次在调用它之前，先确保每个编号的类型与范围合法。

注意，load_records 不会创建缺失文件。完整程序中的“首次创建示例”只是为了方便开始练习，属于另一步。


In [ ]:
def load_records(path):
    json_text = path.read_text(encoding="utf-8")
    return json.loads(json_text)


def decode(token_ids, id_to_token):
    restored = ""

    for token_id in token_ids:
        restored = restored + id_to_token[token_id]

    return restored


### 保留原记录，增加解码结果

`record.copy()` 创建一个新的字典，保留已有字段。随后给新字典增加 decoded_text，不会把这个字段加到原字典上。如果原来已有 decoded_text，生成结果中的这个字段会被重新计算。

这是**浅复制**：内部列表仍可能共享。本课只增加或替换顶层 decoded_text，不修改 token_ids 列表。暂时不需要掌握深复制。


In [ ]:
original = {"line_number": 1, "text": "我", "token_ids": [0]}
result = original.copy()
result["decoded_text"] = decode(result["token_ids"], id_to_token)
print("原记录：", original)
print("输出记录：", result)

mismatch = [{"text": "我", "token_ids": [1]}]
validate_records(mismatch, len(id_to_token))
print("通过结构检查但内容不对应：", mismatch[0]["text"], "→", decode(mismatch[0]["token_ids"], id_to_token))


## 9. 组合成“先检查，后保存”的流程

process_file 接收输入路径、输出路径、词表，返回本次是否成功：True 或 False。

分成三段读：

1. 读取、解析、校验。预期错误发生时打印原因，`return False` 结束当前函数。
2. 所有记录通过后，构造完整输出列表。
3. 保存成功后，才打印“成功”并返回 True。

return False 返回的是 Python 布尔值，不是终端进程的退出码。本课先用它帮助调用者判断结果；命令行退出码留到后面学习。

除三个主要异常外，下面还给文件编码错误 UnicodeDecodeError 和其他文件读写错误 OSError 单独提示。现在先看懂它们与 FileNotFoundError 的写法相同。

同一个 try 的多个 except 会采用第一个匹配项。因此 json.JSONDecodeError 必须放在 ValueError 前面，FileNotFoundError 必须放在更宽的 OSError 前面。这里没有用一个空泛的 except 把所有编程错误隐藏起来。


In [ ]:
def process_file(input_path, output_path, id_to_token):
    # 第一阶段：读取、解析、检查。任一步失败就退出本次函数。
    try:
        records = load_records(input_path)
        checked_records = validate_records(records, len(id_to_token))
    except FileNotFoundError:
        print("读取失败：找不到输入文件：", input_path.resolve())
        return False
    except UnicodeDecodeError:
        print("读取失败：输入文件不是可读取的 UTF-8 文本。请检查文件编码。")
        return False
    except OSError as error:
        print("读取失败：", error)
        return False
    except json.JSONDecodeError as error:
        print("JSON 格式错误：第", error.lineno, "行，第", error.colno, "列：", error.msg)
        return False
    except ValueError as error:
        print("数据检查未通过：", error)
        return False

    # 第二阶段：全部记录检查通过后，才构造输出。
    results = []
    for record in checked_records:
        result = record.copy()
        result["decoded_text"] = decode(record["token_ids"], id_to_token)
        results.append(result)

    # 第三阶段：保存生成结果。写盘失败与输入数据错误分开提示。
    try:
        json_text = json.dumps(results, ensure_ascii=False, indent=2)
        output_path.write_text(json_text, encoding="utf-8")
    except OSError as error:
        print("保存失败：", error)
        print("输出可能没有完整写入，请解决保存问题后重新运行。")
        return False

    print("成功：已检查并保存", len(results), "条记录。")
    print("结果文件：", output_path.resolve())
    return True


## 10. 完整程序：与配套 .py 完全一致

这一单元包含所有依赖，可以单独复制到一个 .py 文件运行，不依赖前面单元留下的函数。

首次运行会创建 4 条示例记录；第 2 条含未知字符标记对应的合法编号 6，第 3 条为空文字。已有 input_records.json 会保留，所以你可以编辑它后重跑。

初始化示例目录与示例文件在处理函数之外，要求当前目录可写。如果要演示“文件不存在”的处理，请调用 process_file 并传入缺失路径；完整程序的启动步骤会为缺失的主输入创建示例。


In [ ]:
# 第 05 课：异常处理与数据检查。
# 运行方法：python lesson_05_errors_and_validation.py
# 只使用 Python 标准库，无需显卡或其他课程文件。
# 数据目录：当前工作目录下的 lesson_05_data。
# input_records.json 缺失时生成示例；已存在时保留原内容。
# 读取、JSON 解析或校验失败时，不会更新 validated_records.json。
# 只检查结构、字段类型与编号范围，不检查 text 与 token_ids 是否对应。
# 初始化示例目录/文件要求当前目录可写；写盘失败不保证旧输出完整。

import json
from pathlib import Path


def decode(token_ids, id_to_token):
    restored = ""

    for token_id in token_ids:
        restored = restored + id_to_token[token_id]

    return restored


def load_records(path):
    json_text = path.read_text(encoding="utf-8")
    return json.loads(json_text)


def validate_records(records, vocab_size):
    if not isinstance(records, list):
        raise ValueError("最外层必须是列表，例如 [{...}, {...}]")

    record_number = 1
    for record in records:
        location = "第 " + str(record_number) + " 条记录"

        if not isinstance(record, dict):
            raise ValueError(location + "必须是字典")
        if "text" not in record:
            raise ValueError(location + "缺少 text 字段")
        if "token_ids" not in record:
            raise ValueError(location + "缺少 token_ids 字段")
        if not isinstance(record["text"], str):
            raise ValueError(location + "的 text 必须是字符串")
        if not isinstance(record["token_ids"], list):
            raise ValueError(location + "的 token_ids 必须是列表")

        token_index = 0
        for token_id in record["token_ids"]:
            token_location = location + "的 token_ids[" + str(token_index) + "]"
            # Python 的 bool 也能通过 int 类型检查，因此先排除它。
            if isinstance(token_id, bool):
                raise ValueError(token_location + "不能是布尔值")
            if not isinstance(token_id, int):
                raise ValueError(token_location + "必须是整数")
            if token_id < 0:
                raise ValueError(token_location + "不能小于 0")
            if token_id >= vocab_size:
                raise ValueError(token_location + "必须小于词表大小 " + str(vocab_size))
            token_index = token_index + 1

        record_number = record_number + 1

    # 检查通过后返回原数据；这个函数不修改、不补齐、不纠正数据。
    return records


def process_file(input_path, output_path, id_to_token):
    # 第一阶段：读取、解析、检查。任一步失败就退出本次函数。
    try:
        records = load_records(input_path)
        checked_records = validate_records(records, len(id_to_token))
    except FileNotFoundError:
        print("读取失败：找不到输入文件：", input_path.resolve())
        return False
    except UnicodeDecodeError:
        print("读取失败：输入文件不是可读取的 UTF-8 文本。请检查文件编码。")
        return False
    except OSError as error:
        print("读取失败：", error)
        return False
    except json.JSONDecodeError as error:
        print("JSON 格式错误：第", error.lineno, "行，第", error.colno, "列：", error.msg)
        return False
    except ValueError as error:
        print("数据检查未通过：", error)
        return False

    # 第二阶段：全部记录检查通过后，才构造输出。
    results = []
    for record in checked_records:
        result = record.copy()
        result["decoded_text"] = decode(record["token_ids"], id_to_token)
        results.append(result)

    # 第三阶段：保存生成结果。写盘失败与输入数据错误分开提示。
    try:
        json_text = json.dumps(results, ensure_ascii=False, indent=2)
        output_path.write_text(json_text, encoding="utf-8")
    except OSError as error:
        print("保存失败：", error)
        print("输出可能没有完整写入，请解决保存问题后重新运行。")
        return False

    print("成功：已检查并保存", len(results), "条记录。")
    print("结果文件：", output_path.resolve())
    return True


id_to_token = ["我", "爱", "学", "习", "模", "型", "<unk>"]

work_dir = Path.cwd() / "lesson_05_data"
work_dir.mkdir(exist_ok=True)
input_path = work_dir / "input_records.json"
output_path = work_dir / "validated_records.json"

sample_records = [
    {"line_number": 1, "text": "我爱学习", "token_ids": [0, 1, 2, 3]},
    {"line_number": 2, "text": "我爱大模型", "token_ids": [0, 1, 6, 4, 5]},
    {"line_number": 3, "text": "", "token_ids": []},
    {"line_number": 4, "text": "学习模型", "token_ids": [2, 3, 4, 5]},
]

# 首次运行便于开始练习；已存在的输入，即使有错误，也不会被替换。
if not input_path.exists():
    sample_json = json.dumps(sample_records, ensure_ascii=False, indent=2)
    input_path.write_text(sample_json, encoding="utf-8")

print("输入文件：", input_path.resolve())
success = process_file(input_path, output_path, id_to_token)
print("本次处理成功：", success)


### 读取结果时，先看本次是否成功

检查失败时，上次输出可能仍然存在。不能因为看到磁盘上有一个 JSON，就认为本次运行成功。下面只在 success 为 True 时读取结果。

初始示例的解码结果应为“我爱学习”“我爱<unk>模型”、空字符串、“学习模型”。输出中的原 text 仍保留“大”，因为它是单独保存的原文；编号本身没有恢复未知字符的能力。


In [ ]:
if success:
    saved_records = load_records(output_path)
    for record in saved_records:
        print(record)
else:
    print("本次失败，先修复输入；已有输出不能代表这次输入的处理结果。")


## 11. 遇到问题时，按这个顺序检查

| 提示 | 先查看什么 |
| --- | --- |
| 找不到输入文件 | 程序打印的完整路径；当前工作目录是否与你预想的一致 |
| JSON 格式错误 | 提示的 JSON 行、列附近；缺引号、缺括号、多逗号都可能导致失败 |
| 最外层必须是列表 | 文件外面是否需要方括号；单个字典也是合法 JSON，但不符合本课约定 |
| 缺少字段 | text / token_ids 是否拼写一致 |
| 编号类型或范围错误 | 是否写成字符串、布尔值、浮点数、负数或 7 以上的数字 |
| 保存失败 | 输出路径是否可写，父目录是否存在 |

一次报告第一个问题。修复后再运行，如果还有下一处问题，就继续处理。没有出错不代表内容正确：本课的检查不会验证文字含义，也不会证明 text 与 token_ids 来自同一次编码。


## 12. 练习 A：区分“格式错误”和“结构错误”

对下面两段 JSON 文本，依次执行 json.loads，然后检查最外层是否为 list。分别捕获 json.JSONDecodeError 和 ValueError，并打印清楚是哪一层失败。

第一段应在解析阶段失败；第二段应解析成功，但被列表检查拒绝。把它们都修成可通过的数据，再运行一次。


In [ ]:
exercise_a_texts = [
    '[{"text": "我", "token_ids": [0]},]',
    '{"text": "我", "token_ids": [0]}',
]
# 在这里写你的循环、try/except 和最外层列表检查。
# 写完后，再补一段修正过的 JSON 文本并验证。


## 13. 练习 B：根据定位信息修复记录

分别把下面的单条记录装进列表，调用 validate_records。每一次调用都用 try/except ValueError 打印问题，保证后面的例子也会运行。

修复四条记录，让它们都表达 `{"text": "我爱", "token_ids": [0, 1]}`。然后再验证未知字符记录 `{"text": "大", "token_ids": [6]}`，解释为什么它合法。

**可选追问：** 把编号改成 -1、"1"、1.0，各会在哪个检查中被拒绝？


In [ ]:
exercise_b_records = [
    {"token_ids": [0, 1]},
    {"text": "我爱", "token_ids": "0, 1"},
    {"text": "我爱", "token_ids": [0, True]},
    {"text": "我爱", "token_ids": [0, 7]},
]
# 每次传入 [record]，因为校验函数要求最外层是列表。
# 在这里写检查、错误提示和修复后的数据。


## 14. 练习 C：失败时保留上次结果

在独立的 `lesson_05_data/my_exercise_c` 目录里练习，输入、输出使用两个不同文件。不要改主示例输入。

1. 写入合法输入 `[{"text": "我", "token_ids": [0]}]`，调用 process_file，保存成功。
2. 读取输出内容，存入 before。
3. 把这份练习输入改成 `[{"text": "我", "token_ids": [7]}]`，再次调用。
4. 读取输出内容，存入 after；用 `before == after` 判断是否保持不变。`==` 比较两个值是否相等。
5. 修正输入为 `[{"text": "学习", "token_ids": [2, 3]}]`，处理成功后查看新的 decoded_text。

你应看到“成功 → 失败且旧输出不变 → 修正后成功”。保留旧文件不等于当前输入处理成功，要同时观察本次返回值。


In [ ]:
# 在这里完成练习 C。
# 先创建专用目录，再选择不同的输入、输出路径。
# 用 json.dumps + write_text 写入练习输入。
# 用 process_file(...) 的返回值判断本次是否成功。


---

## 参考答案：先完成自己的实现再往下运行

下方三个答案各自包含所需定义，可独立复制运行。参考答案 C 使用 generated_exercise_c 目录，重复运行会重新生成其中的演示数据，不触碰你的 my_exercise_c 或主输入文件。

### A：分清解析与结构检查


In [ ]:
import json
from pathlib import Path

answer_a_texts = [
    '[{"text": "我", "token_ids": [0]},]',
    '{"text": "我", "token_ids": [0]}',
    '[{"text": "我", "token_ids": [0]}]',
]

for json_text in answer_a_texts:
    try:
        records = json.loads(json_text)
        if not isinstance(records, list):
            raise ValueError("最外层必须是列表")
        print("解析与最外层检查都通过：", records)
    except json.JSONDecodeError as error:
        print("解析失败：第", error.lineno, "行，第", error.colno, "列")
    except ValueError as error:
        print("结构检查失败：", error)


### B：识别四类问题，并修复

注意，本题每次单独检查一个只有一条记录的列表，所以记录序号始终是“第 1 条”。如果一次传入整个多条列表，遇到第一个错误就会停止，不会自动收集全部错误。


In [ ]:
def validate_records(records, vocab_size):
    if not isinstance(records, list):
        raise ValueError("最外层必须是列表，例如 [{...}, {...}]")

    record_number = 1
    for record in records:
        location = "第 " + str(record_number) + " 条记录"

        if not isinstance(record, dict):
            raise ValueError(location + "必须是字典")
        if "text" not in record:
            raise ValueError(location + "缺少 text 字段")
        if "token_ids" not in record:
            raise ValueError(location + "缺少 token_ids 字段")
        if not isinstance(record["text"], str):
            raise ValueError(location + "的 text 必须是字符串")
        if not isinstance(record["token_ids"], list):
            raise ValueError(location + "的 token_ids 必须是列表")

        token_index = 0
        for token_id in record["token_ids"]:
            token_location = location + "的 token_ids[" + str(token_index) + "]"
            # Python 的 bool 也能通过 int 类型检查，因此先排除它。
            if isinstance(token_id, bool):
                raise ValueError(token_location + "不能是布尔值")
            if not isinstance(token_id, int):
                raise ValueError(token_location + "必须是整数")
            if token_id < 0:
                raise ValueError(token_location + "不能小于 0")
            if token_id >= vocab_size:
                raise ValueError(token_location + "必须小于词表大小 " + str(vocab_size))
            token_index = token_index + 1

        record_number = record_number + 1

    # 检查通过后返回原数据；这个函数不修改、不补齐、不纠正数据。
    return records


id_to_token = ["我", "爱", "学", "习", "模", "型", "<unk>"]

answer_b_bad = [
    {"token_ids": [0, 1]},
    {"text": "我爱", "token_ids": "0, 1"},
    {"text": "我爱", "token_ids": [0, True]},
    {"text": "我爱", "token_ids": [0, 7]},
]
for record in answer_b_bad:
    try:
        validate_records([record], len(id_to_token))
    except ValueError as error:
        print("发现问题：", error)

answer_b_fixed = [
    {"text": "我爱", "token_ids": [0, 1]},
    {"text": "我爱", "token_ids": [0, 1]},
    {"text": "我爱", "token_ids": [0, 1]},
    {"text": "我爱", "token_ids": [0, 1]},
    {"text": "大", "token_ids": [6]},
]
print("修复后通过：", validate_records(answer_b_fixed, len(id_to_token)))


### C：验证失败后，生成文件保持原内容

参考答案使用单独的演示目录，第一次主动写入正常输入，第二次写入错误输入，最后修复。before == after 比较的是两次读取的文件文字。


In [ ]:
import json
from pathlib import Path


def decode(token_ids, id_to_token):
    restored = ""

    for token_id in token_ids:
        restored = restored + id_to_token[token_id]

    return restored


def load_records(path):
    json_text = path.read_text(encoding="utf-8")
    return json.loads(json_text)


def validate_records(records, vocab_size):
    if not isinstance(records, list):
        raise ValueError("最外层必须是列表，例如 [{...}, {...}]")

    record_number = 1
    for record in records:
        location = "第 " + str(record_number) + " 条记录"

        if not isinstance(record, dict):
            raise ValueError(location + "必须是字典")
        if "text" not in record:
            raise ValueError(location + "缺少 text 字段")
        if "token_ids" not in record:
            raise ValueError(location + "缺少 token_ids 字段")
        if not isinstance(record["text"], str):
            raise ValueError(location + "的 text 必须是字符串")
        if not isinstance(record["token_ids"], list):
            raise ValueError(location + "的 token_ids 必须是列表")

        token_index = 0
        for token_id in record["token_ids"]:
            token_location = location + "的 token_ids[" + str(token_index) + "]"
            # Python 的 bool 也能通过 int 类型检查，因此先排除它。
            if isinstance(token_id, bool):
                raise ValueError(token_location + "不能是布尔值")
            if not isinstance(token_id, int):
                raise ValueError(token_location + "必须是整数")
            if token_id < 0:
                raise ValueError(token_location + "不能小于 0")
            if token_id >= vocab_size:
                raise ValueError(token_location + "必须小于词表大小 " + str(vocab_size))
            token_index = token_index + 1

        record_number = record_number + 1

    # 检查通过后返回原数据；这个函数不修改、不补齐、不纠正数据。
    return records


def process_file(input_path, output_path, id_to_token):
    # 第一阶段：读取、解析、检查。任一步失败就退出本次函数。
    try:
        records = load_records(input_path)
        checked_records = validate_records(records, len(id_to_token))
    except FileNotFoundError:
        print("读取失败：找不到输入文件：", input_path.resolve())
        return False
    except UnicodeDecodeError:
        print("读取失败：输入文件不是可读取的 UTF-8 文本。请检查文件编码。")
        return False
    except OSError as error:
        print("读取失败：", error)
        return False
    except json.JSONDecodeError as error:
        print("JSON 格式错误：第", error.lineno, "行，第", error.colno, "列：", error.msg)
        return False
    except ValueError as error:
        print("数据检查未通过：", error)
        return False

    # 第二阶段：全部记录检查通过后，才构造输出。
    results = []
    for record in checked_records:
        result = record.copy()
        result["decoded_text"] = decode(record["token_ids"], id_to_token)
        results.append(result)

    # 第三阶段：保存生成结果。写盘失败与输入数据错误分开提示。
    try:
        json_text = json.dumps(results, ensure_ascii=False, indent=2)
        output_path.write_text(json_text, encoding="utf-8")
    except OSError as error:
        print("保存失败：", error)
        print("输出可能没有完整写入，请解决保存问题后重新运行。")
        return False

    print("成功：已检查并保存", len(results), "条记录。")
    print("结果文件：", output_path.resolve())
    return True


id_to_token = ["我", "爱", "学", "习", "模", "型", "<unk>"]

answer_c_dir = Path.cwd() / "lesson_05_data" / "generated_exercise_c"
answer_c_dir.mkdir(parents=True, exist_ok=True)
answer_c_input = answer_c_dir / "input_records.json"
answer_c_output = answer_c_dir / "validated_records.json"

good_records = [{"text": "我", "token_ids": [0]}]
answer_c_input.write_text(json.dumps(good_records, ensure_ascii=False, indent=2), encoding="utf-8")
first_ok = process_file(answer_c_input, answer_c_output, id_to_token)
before = answer_c_output.read_text(encoding="utf-8")

bad_records = [{"text": "我", "token_ids": [7]}]
answer_c_input.write_text(json.dumps(bad_records, ensure_ascii=False, indent=2), encoding="utf-8")
second_ok = process_file(answer_c_input, answer_c_output, id_to_token)
after = answer_c_output.read_text(encoding="utf-8")
print("第一次成功：", first_ok)
print("第二次成功：", second_ok)
print("失败后旧输出保持不变：", before == after)

repaired_records = [{"text": "学习", "token_ids": [2, 3]}]
answer_c_input.write_text(json.dumps(repaired_records, ensure_ascii=False, indent=2), encoding="utf-8")
third_ok = process_file(answer_c_input, answer_c_output, id_to_token)
print("修复后成功：", third_ok)
print("修复后的输出：", load_records(answer_c_output))


`mkdir(parents=True, exist_ok=True)` 中的 parents=True 表示需要时连同尚不存在的父目录一起创建。这样参考答案 C 单独运行也能准备好演示目录。

## 15. 完成本课的标准

- 能说明 try、except、raise、return False 分别做什么。
- 能区分“JSON 文本非法”与“JSON 合法但数据结构不符合约定”。
- 能说清为什么先检查类型，再访问字段、循环或比较编号。
- 能解释为什么 6 合法，而 True、-1、7 不合法。
- 能写出一个小检查函数，例如要求 text 字段存在且为字符串。
- 能完成练习 C，既检查返回值，也查看输出是否更新。
- 能举出通过本课检查但内容仍不正确的例子。

建议把练习 A 的错误提示、练习 B 的修复代码，以及练习 C 的三次返回值发来核对。不要以“运行全部没有红色错误”代替这些练习。

## 与后续大模型实践的联系

以后读取训练样本、解析模型的结构化输出、检查 Agent 工具参数，也需要先明确约定，再验证输入，并决定失败时如何处理。今天练的是这一小段通用程序逻辑；实际模型输出的内容正确性、任务成功率和工具权限还要分别评估。

**下一课：第 06 课，模块、程序入口与小型项目组织。** 把已有函数放进不同 .py 文件，理解 import 与直接运行的区别，让后续代码可以复用。

## 官方参考资料

- [Python 3.12：错误和异常](https://docs.python.org/zh-cn/3.12/tutorial/errors.html)
- [Python 3.12：json 模块与 JSONDecodeError](https://docs.python.org/zh-cn/3.12/library/json.html)
- [Python 3.12：内置函数 isinstance、str 与 bool](https://docs.python.org/zh-cn/3.12/library/functions.html)
- [Python 3.12：字典 copy 与比较运算](https://docs.python.org/zh-cn/3.12/library/stdtypes.html)
